# MPMS Magnetometry Analysis

This notebook processes raw magnetometry data from a Magnetic Property Measurement System (MPMS).  
It covers two related analyses:

1. **Volume fraction susceptibility** — computes the Meissner shielding fraction $4\pi\chi/V$ to quantify superconducting volume fraction as a function of temperature.
2. **Normalized susceptibility and $T_c$ identification** — parses raw `.dat` files, computes $\chi = M/H$, normalizes, and marks the superconducting transition temperature $T_c$.

**Data format:** Quantum Design MPMS `.dat` files (CSV with a 44-line header).  
**Place your data files in the `data/` folder** before running.


## 1. Imports


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
from matplotlib.ticker import AutoMinorLocator
%matplotlib inline


## 2. Plot Formatting Helpers

These functions apply consistent publication-style formatting to all plots:
inward-facing ticks on all four sides, minor ticks, and a clean legend box.


In [ ]:
def format_my_plot(figsize=(5, 5)):
    """Set figure size and global rcParams for publication-style axes."""
    plt.gcf().set_size_inches(*figsize)
    plt.rcParams.update({'axes.linewidth': 1.2, 'font.size': 14})
    plt.locator_params(axis='y', nbins=6)
    plt.tick_params(axis='y', left=True, right=True, direction='in', length=13, width=1.2)
    plt.tick_params(axis='x', bottom=True, top=True, direction='in', length=13, width=1.2)
    plt.gca().yaxis.set_minor_locator(AutoMinorLocator(2))
    plt.gca().xaxis.set_minor_locator(AutoMinorLocator(2))
    plt.tick_params(which='minor', direction='in', left=True, right=True,
                    bottom=True, top=True, length=7, width=1.2)
    plt.gca().set_aspect(1.0 / plt.gca().get_data_ratio())
    plt.tight_layout()

def format_legend():
    """Style the legend with a clean black border and white background."""
    leg = plt.legend(framealpha=1)
    leg.get_frame().set_edgecolor('black')
    leg.get_frame().set_facecolor('white')
    leg.get_frame().set_linewidth(1.1)

def format_my_ticks():
    """Apply tick formatting without changing figure size (use after format_my_plot)."""
    plt.locator_params(axis='y', nbins=6)
    plt.tick_params(axis='y', left=True, right=True, direction='in', length=13, width=1.2)
    plt.tick_params(axis='x', bottom=True, top=True, direction='in', length=13, width=1.2)
    plt.gca().yaxis.set_minor_locator(AutoMinorLocator(2))
    plt.gca().xaxis.set_minor_locator(AutoMinorLocator(2))
    plt.tick_params(which='minor', direction='in', left=True, right=True,
                    bottom=True, top=True, length=7, width=1.2)


## 3. Data Parsing

### 3a. Volume Fraction Analysis (multi-field `.dat` files)

`sort_dat` reads a Quantum Design MPMS `.dat` file and groups measurements by applied field.  
`plot_tc` computes the volume fraction susceptibility:

$$\chi_{\text{vol}} = \frac{M}{H} \cdot \frac{4\pi}{V}$$

where $V = m / \rho$ is the sample volume. For powder samples a packing density correction of 0.7 is applied.


In [ ]:
def sort_dat(filename):
    """
    Read a Quantum Design MPMS .dat file and group rows by applied field.

    Parameters
    ----------
    filename : str
        Path to the .dat file (relative to this notebook).

    Returns
    -------
    dict
        Keys are rounded integer field values (Oe); values are DataFrames.
    """
    dat = pd.read_csv(filename, header=44, skipinitialspace=False)
    dat['int Field (Oe)'] = dat['Magnetic Field (Oe)'].apply(round)
    groups = dat.groupby('int Field (Oe)')
    return {field: groups.get_group(field) for field in groups.groups}


def parse(raw):
    """Extract columns from a field-grouped DataFrame."""
    chi = raw['Moment (emu)'] / raw['Magnetic Field (Oe)']
    return (
        raw['Temperature (K)'],
        chi,
        raw['Moment (emu)'],
        raw['Magnetic Field (Oe)'],
        raw['Time Stamp (sec)'],
        raw['AC Susceptibility (emu/Oe)'],
        raw['AC Drive (Oe)'],
    )


def plot_tc(mass_mg, density_gcc, filepath, fields=(10, 10000), powder=False):
    """
    Compute volume fraction susceptibility at two applied fields.

    Parameters
    ----------
    mass_mg : float
        Sample mass in milligrams.
    density_gcc : float
        Sample density in g/cm³.
    filepath : str
        Path to the MPMS .dat file.
    fields : tuple of int
        Two integer field values (Oe) to extract. Default: (10, 10000).
        Typically a low field for the Meissner transition and a high field
        for the paramagnetic normal state.
    powder : bool
        If True, divide volume by 0.7 to correct for powder packing density.

    Returns
    -------
    [T_low, chi_vol_low], [T_high, chi_vol_high]
        Temperature arrays and volume fraction susceptibility arrays
        for the low and high field measurements.
    """
    frames = sort_dat(filepath)
    vol = (mass_mg / 1000) / density_gcc          # cm³
    if powder:
        vol /= 0.7                                 # packing density correction

    T1, chi1, *_ = parse(frames[fields[0]].sort_values('Temperature (K)'))
    T2, chi2, *_ = parse(frames[fields[1]].sort_values('Temperature (K)'))

    chi_vol1 = chi1 * 4 * np.pi / vol
    chi_vol2 = chi2 * 4 * np.pi / vol
    return [T1, chi_vol1], [T2, chi_vol2]


### 3b. Single-file Parser (arbitrary field `.dat` files)

`parse_dat` handles files where the field may vary or is recorded in the header.
It returns temperature, moment, and field arrays directly.


In [ ]:
def parse_dat(file_path):
    """
    Parse a Quantum Design MPMS .dat file with an unknown or variable field.

    Searches the 44-line header for a field value first.
    Falls back to reading the field column from the data block.

    Parameters
    ----------
    file_path : str
        Path to the .dat file.

    Returns
    -------
    temps : ndarray
        Temperature values in Kelvin.
    moments : ndarray
        Magnetic moment values in emu.
    field_value : float or None
        Applied field in Oe (None if it could not be determined).
    """
    try:
        with open(file_path, 'r') as f:
            header_lines = [next(f) for _ in range(44)]

        # Try to extract field from header
        field_value = None
        for line in header_lines:
            if 'field' in line.lower() and ':' in line:
                try:
                    field_value = float(line.split(':')[1].strip().split()[0])
                    break
                except (ValueError, IndexError):
                    continue

        df = pd.read_csv(file_path, header=44, engine='python', skipinitialspace=True)
        df.columns = df.columns.str.strip()

        temp_col   = next((c for c in df.columns if 'temperature' in c.lower()), None)
        moment_col = next((c for c in df.columns if 'moment' in c.lower() and 'err' not in c.lower()), None)
        field_col  = next((c for c in df.columns if 'field' in c.lower()), None)

        if not temp_col or not moment_col:
            raise ValueError(f'Could not find Temperature or Moment columns in {file_path}')

        # Fall back to field column if header search failed
        if field_value is None and field_col:
            unique_fields = df[field_col].dropna().unique()
            field_value = unique_fields[0] if len(unique_fields) == 1 else unique_fields.mean()

        df = df[[temp_col, moment_col]].dropna().drop_duplicates(subset=temp_col)
        return df[temp_col].values, df[moment_col].values, field_value

    except Exception as e:
        print(f'[!] Could not read {file_path}: {e}')
        return None, None, None


## 4. Volume Fraction Plot

Plot $4\pi\chi/V$ vs temperature to show the Meissner shielding fraction.  
A value near $-1$ at low temperature indicates full superconducting volume fraction.

**Edit the cell below** to point to your data file and supply the correct mass and density.


In [ ]:
# ── User inputs ───────────────────────────────────────────────
FILEPATH  = 'data/ZrV2_sample.dat'   # path to your MPMS .dat file
MASS_MG   = 1.45                # sample mass in mg
DENSITY   = 6.23                # density in g/cm³
FIELDS    = (10, 10000)         # (low field Oe, high field Oe)
POWDER    = False
# ─────────────────────────────────────────────────────────────

low_field, high_field = plot_tc(MASS_MG, DENSITY, FILEPATH,
                                fields=FIELDS, powder=POWDER)

plt.figure(dpi=150, figsize=(6, 4))
plt.plot(*high_field, '.-', markevery=30, color='purple', label=f'{FIELDS[1]} Oe')
plt.plot(*low_field,  '.-', markevery=30, color='teal',   label=f'{FIELDS[0]} Oe')
plt.ylim(-1.2, 0.1)
plt.xlim(0, 300)
plt.xlabel('Temperature (K)')
plt.ylabel(r'$\chi_{\mathrm{vol}}\;(4\pi\;\mathrm{emu}\;\mathrm{Oe}^{-1}\;\mathrm{cm}^{-3})$')
plt.title('Volume Fraction Susceptibility vs Temperature')
format_my_ticks()
format_legend()
plt.tight_layout()
plt.show()


## 5. Curie-Weiss Fit (Normal State)

Above $T_c$, the susceptibility of many superconductors follows a Curie-Weiss law:

$$\chi(T) = \frac{C}{T - \theta_{\mathrm{CW}}} + \chi_0$$

Fitting this to the high-temperature data extracts the Curie constant $C$,  
the Weiss temperature $\theta_{\mathrm{CW}}$, and a temperature-independent offset $\chi_0$.


In [ ]:
def chi_curie_weiss(T, C, theta_CW, chi0):
    """Curie-Weiss model for magnetic susceptibility."""
    return C / (T - theta_CW) + chi0

T_high, chi_high = high_field

# Fit only above Tc (adjust threshold as needed)
T_MIN_FIT = 17   # K
mask = T_high > T_MIN_FIT
T_fit   = T_high[mask]
chi_fit = chi_high[mask]

popt, _ = curve_fit(chi_curie_weiss, T_fit, chi_fit, p0=[1e-3, 0, 0])
C_fit, theta_fit, chi0_fit = popt

print(f'Curie constant     C        = {C_fit:.4e} emu·K/Oe')
print(f'Weiss temperature  θ_CW     = {theta_fit:.2f} K')
print(f'Offset             χ₀       = {chi0_fit:.4e} emu/Oe')

plt.figure(dpi=150, figsize=(6, 4))
plt.plot(T_fit, chi_fit, '.-', markevery=30, color='purple', label=r'$\chi$ data')
plt.plot(T_fit, chi_curie_weiss(T_fit, *popt), '--', color='blue',
         label=rf'Fit: $C={C_fit:.2e}$, $\theta={theta_fit:.1f}$ K')
plt.xlabel('Temperature (K)')
plt.ylabel(r'$\chi\;(\mathrm{emu}\;\mathrm{Oe}^{-1})$')
plt.title('Curie-Weiss Fit (Normal State)')
format_my_ticks()
format_legend()
plt.tight_layout()
plt.show()


## 6. Normalized Susceptibility and $T_c$ Identification

Parse a raw `.dat` file, compute $\chi = M/H$, normalize to the maximum absolute value,  
and mark the superconducting transition temperature $T_c$ with a vertical line.


In [ ]:
# ── User inputs ───────────────────────────────────────────────
FILEPATH_RAW = 'data/ZrV2_sample.dat'
LABEL        = 'Sample'
T_MAX        = 20      # K — upper temperature limit for the plot
TC           = None    # set to a float (e.g. 8.8) to draw a Tc marker
# ─────────────────────────────────────────────────────────────

temps, moments, H = parse_dat(FILEPATH_RAW)

if temps is not None:
    chi    = moments / H
    norm_chi = chi / np.max(np.abs(chi))

    mask = temps <= T_MAX
    T_plot   = temps[mask]
    chi_plot = norm_chi[mask]

    plt.figure(figsize=(6, 6), dpi=150)
    plt.plot(T_plot, chi_plot, color='#073763', linestyle='-', label=LABEL)

    if TC is not None:
        plt.axvline(x=TC, color='red', linestyle='--', linewidth=1.5,
                    label=f'$T_c = {TC}$ K')
        plt.text(TC + 0.2, 0.05, r'$T_c$', color='red', fontsize=12)

    plt.xlabel('Temperature (K)')
    plt.ylabel(r'Normalized $\chi$ (arb. units)')
    plt.title(f'Superconducting Transition — {LABEL}')
    format_my_plot()
    format_my_ticks()
    format_legend()
    plt.tight_layout()
    plt.show()
